# NOVA RETAIL — Shadow Inventory & Ghost SKUs

## Análisis Forense de Inventario Invisible
**Proyecto:** Diagnóstico de Prevención de Pérdidas  
**Autor:** Denz One  
**Fase:** Identificación de vulnerabilidades estructurales en el catálogo legacy  
**Objetivo:** Detectar productos presentes en SAP pero invisibles para AS400, cuantificar su exposición económica y evaluar su relación con los puntos ciegos de control del inventario.

---

### Propósito de este notebook
Este notebook analiza una de las vulnerabilidades más críticas del entorno NOVA RETAIL:

- productos que existen en SAP pero no en AS400
- mercancía de alto valor que puede circular sin trazabilidad completa
- riesgo de pérdida material facilitado por opacidad estructural de catálogo

### Preguntas clave
1. ¿Cuántos SKUs son invisibles para AS400?
2. ¿Qué valor económico concentran?
3. ¿Qué categorías o marcas dominan esa invisibilidad?
4. ¿Qué tan defendible es tratar este fenómeno como una vulnerabilidad material?

### Nota del analista
La invisibilidad de catálogo no prueba por sí sola desvío intencional.  
Pero sí crea una condición operativa donde la mercancía puede moverse con menor capacidad de control, conciliación y auditoría.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_PATH = Path("../data/raw")

products_sap = pd.read_csv(DATA_PATH / "products_sap.csv")
products_as400 = pd.read_csv(DATA_PATH / "products_as400.csv")
dispatches = pd.read_csv(DATA_PATH / "dispatches.csv")
receptions = pd.read_csv(DATA_PATH / "receptions.csv")

print("Datasets cargados correctamente.")
print("products_sap:", products_sap.shape)
print("products_as400:", products_as400.shape)
print("dispatches:", dispatches.shape)
print("receptions:", receptions.shape)

In [ ]:
apple_ghosts = products_as400[products_as400["is_ghost"] == True].copy()

print(f"Total Ghost SKUs: {len(apple_ghosts)}")
apple_ghosts.head()

In [ ]:
ghost_sap_refs = apple_ghosts["sku_sap_reference"].dropna().tolist()

sap_ghost_details = products_sap[
    products_sap["sku_sap"].isin(ghost_sap_refs)
].copy()

sap_ghost_details.head(20)

In [ ]:
ghost_sap_refs = apple_ghosts["sku_sap_reference"].dropna().tolist()

sap_ghost_details = products_sap[
    products_sap["sku_sap"].isin(ghost_sap_refs)
].copy()

sap_ghost_details.head(20)

In [ ]:
sap_ghost_details[[
    "sku_sap",
    "description_sap",
    "category_sap",
    "brand",
    "price_sap"
]].sort_values("price_sap", ascending=False).head(20)

In [ ]:
ghost_summary = pd.DataFrame({
    "indicador": [
        "Total Ghost SKUs",
        "Valor promedio por SKU",
        "Valor máximo por SKU",
        "Valor mínimo por SKU",
        "Valor total estimado del catálogo invisible"
    ],
    "valor": [
        len(sap_ghost_details),
        sap_ghost_details["price_sap"].mean(),
        sap_ghost_details["price_sap"].max(),
        sap_ghost_details["price_sap"].min(),
        sap_ghost_details["price_sap"].sum()
    ]
})

ghost_summary

In [ ]:
sap_ghost_details["category_sap"].value_counts()

In [ ]:
sap_ghost_details["brand"].value_counts()

In [ ]:
ghost_dispatches = dispatches[
    dispatches["sku"].isin(ghost_sap_refs)
].copy()

print("Despachos de Ghost SKUs:", len(ghost_dispatches))
ghost_dispatches.head(20)

In [ ]:
ghost_dispatches["route"].value_counts()

In [ ]:
ghost_dispatches["store_id"].nunique()

In [ ]:
ghost_dispatch_summary = ghost_dispatches.groupby("store_id").agg(
    total_despachos=("dispatch_id", "count"),
    total_unidades=("quantity_dispatched", "sum")
).reset_index().sort_values("total_unidades", ascending=False)

ghost_dispatch_summary.head(20)

In [ ]:
ghost_dispatch_value = ghost_dispatches.merge(
    products_sap[["sku_sap", "price_sap", "description_sap", "brand", "category_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="left"
)

ghost_dispatch_value["dispatch_value_mxn"] = (
    ghost_dispatch_value["quantity_dispatched"] * ghost_dispatch_value["price_sap"]
)

ghost_dispatch_value[[
    "dispatch_id",
    "store_id",
    "sku",
    "description_sap",
    "quantity_dispatched",
    "price_sap",
    "dispatch_value_mxn",
    "route",
    "authorized_by"
]].head(20)

In [ ]:
ghost_value_summary = pd.DataFrame({
    "indicador": [
        "Despachos de Ghost SKUs",
        "Tiendas alcanzadas",
        "Rutas involucradas",
        "Unidades totales despachadas",
        "Valor económico total despachado (MXN)"
    ],
    "valor": [
        len(ghost_dispatch_value),
        ghost_dispatch_value["store_id"].nunique(),
        ghost_dispatch_value["route"].nunique(),
        ghost_dispatch_value["quantity_dispatched"].sum(),
        ghost_dispatch_value["dispatch_value_mxn"].sum()
    ]
})

ghost_value_summary

In [ ]:
ghost_dispatch_value.groupby("route").agg(
    despachos=("dispatch_id", "count"),
    unidades=("quantity_dispatched", "sum"),
    valor_total_mxn=("dispatch_value_mxn", "sum")
).sort_values("valor_total_mxn", ascending=False)

In [ ]:
top_severe_store_ids = [156, 47, 15, 83, 29, 134, 67, 91, 112, 171]

ghost_to_severe = ghost_dispatch_value[
    ghost_dispatch_value["store_id"].isin(top_severe_store_ids)
].copy()

ghost_to_severe_summary = ghost_to_severe.groupby("store_id").agg(
    despachos_ghost=("dispatch_id", "count"),
    unidades_ghost=("quantity_dispatched", "sum"),
    valor_ghost_mxn=("dispatch_value_mxn", "sum")
).reset_index().sort_values("valor_ghost_mxn", ascending=False)

ghost_to_severe_summary

In [ ]:
ghost_to_severe_summary["store_id"] = ghost_to_severe_summary["store_id"].astype(str)

fig = px.bar(
    ghost_to_severe_summary.sort_values("valor_ghost_mxn", ascending=True),
    x="valor_ghost_mxn",
    y="store_id",
    orientation="h",
    text="valor_ghost_mxn",
    title="Valor de Ghost SKUs despachados a tiendas severas"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside",
    marker_color="#EF4444",
    hovertemplate="<b>Tienda %{y}</b><br>Valor Ghost: $%{x:,.0f} MXN<extra></extra>"
)

fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white"),
    height=650,
    xaxis_title="Valor económico despachado (MXN)",
    yaxis_title="Store ID"
)

fig.show()

In [ ]:
ghost_to_severe_summary["store_id"] = ghost_to_severe_summary["store_id"].astype(str)

fig = px.scatter(
    ghost_to_severe_summary,
    x="unidades_ghost",
    y="valor_ghost_mxn",
    size="despachos_ghost",
    text="store_id",
    color="valor_ghost_mxn",
    color_continuous_scale=[
        [0.00, "#3B82F6"],
        [0.35, "#06B6D4"],
        [0.70, "#8B5CF6"],
        [1.00, "#EF4444"]
    ],
    title="Exposición de Ghost SKUs en tiendas severas"
)

fig.update_traces(
    textposition="top center",
    marker=dict(line=dict(color="rgba(255,255,255,0.25)", width=1.2)),
    hovertemplate=(
        "<b>Tienda %{text}</b><br>" +
        "Unidades Ghost: %{x}<br>" +
        "Valor Ghost: $%{y:,.0f} MXN<br>" +
        "Despachos Ghost: %{marker.size}<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white", size=14),
    height=700,
    xaxis_title="Unidades Ghost despachadas",
    yaxis_title="Valor económico Ghost (MXN)",
    coloraxis_showscale=False
)

fig.show()

## Conclusión ejecutiva

Hemos identificado un flujo material de aproximadamente **$49.9M MXN** en mercancía movilizada bajo SKUs sin trazabilidad completa en el entorno legacy, utilizando activamente la infraestructura oficial de la compañía. Esta exposición económica se superpone con las tiendas de mayor severidad operativa, lo que convierte una brecha de catálogo y control en una vulnerabilidad patrimonial y de cumplimiento de alta materialidad. La recomendación inmediata es intervenir este circuito de inventario de forma focalizada para contener la pérdida, restaurar trazabilidad y preparar una segunda fase de revisión investigativa bajo resguardo formal de evidencia.

## Conclusión ejecutiva

El análisis confirma que la opacidad de catálogo ya no es un problema técnico aislado, sino un riesgo operativo material: aproximadamente **$49.9M MXN** en mercancía se han movilizado bajo SKUs sin trazabilidad completa en AS400. Este flujo coincide con las tiendas de mayor severidad de discrepancia, lo que indica que las fallas de control y la pérdida material convergen en los mismos nodos de riesgo. La prioridad inmediata debe ser contener ese circuito, recuperar visibilidad sobre los activos premium y escalar a una segunda fase de validación forense.

## Nota estratégica — Respuesta al contraargumento de Logística

### Contraargumento esperado
Una objeción previsible desde Logística sería afirmar que la presencia de Ghost SKUs en múltiples rutas y tiendas demuestra que el hallazgo corresponde únicamente a un problema sistémico de maestro de datos, y no a una vulnerabilidad localizada.

### Respuesta analítica
Ese argumento es parcialmente correcto, pero insuficiente.

Sí existe una debilidad estructural de trazabilidad en el entorno legacy:
- los Ghost SKUs aparecen en múltiples rutas
- la opacidad del catálogo no está limitada a una sola tienda
- la falla base de control es corporativa

Sin embargo, el hallazgo principal no descansa únicamente en la existencia de esos SKUs invisibles.

La señal crítica emerge de la **convergencia de variables** que no están distribuidas de manera homogénea en toda la red:

- discrepancias materiales de recepción entre **13% y 20.5%**
- concentración completa de tiendas severas en la **Ruta Norte**
- presencia de recepciones en la franja de las **05:00**, ausente en las tiendas de comportamiento normal
- superposición entre tiendas severas y flujo de Ghost SKUs

### Conclusión metodológica
Por tanto, el análisis no sostiene que la Ruta Norte sea la única parte afectada por la debilidad del sistema.  
Lo que sostiene es algo más preciso:

> La falla de control es sistémica, pero su manifestación operativa más severa y económicamente relevante aparece localizada en un corredor específico.

### Implicación de negocio
Esto significa que la respuesta no debe ser binaria (“todo es un problema de sistema” vs “todo es un problema local”), sino secuencial:

1. reconocer la debilidad estructural corporativa
2. intervenir primero donde esa debilidad coincide con pérdida material y comportamiento atípico
3. escalar después la corrección del control a toda la red

### Riesgo de una lectura equivocada
Si se trata todo el problema como puramente sistémico, se diluye la capacidad de priorización y se corre el riesgo de distribuir recursos de forma homogénea donde el impacto no es homogéneo.

En otras palabras:

> El sistema puede estar mal en toda la empresa; la pérdida material no parece estarlo.

In [ ]:
resumen_shadow_inventory = pd.DataFrame({
    "indicador": [
        "Ghost SKUs identificados",
        "Valor total del catálogo invisible (MXN)",
        "Despachos de Ghost SKUs",
        "Tiendas alcanzadas",
        "Rutas involucradas",
        "Unidades totales despachadas",
        "Valor económico total movilizado (MXN)",
        "Tiendas severas expuestas a Ghost SKUs"
    ],
    "valor": [
        len(sap_ghost_details),
        round(sap_ghost_details["price_sap"].sum(), 2),
        len(ghost_dispatch_value),
        ghost_dispatch_value["store_id"].nunique(),
        ghost_dispatch_value["route"].nunique(),
        int(ghost_dispatch_value["quantity_dispatched"].sum()),
        round(ghost_dispatch_value["dispatch_value_mxn"].sum(), 2),
        ghost_to_severe_summary["store_id"].nunique()
    ]
})

resumen_shadow_inventory

## Conclusiones de la fase Shadow Inventory & Ghost SKUs

### Hallazgos principales
- Se identificó un subconjunto de SKUs visibles en SAP pero invisibles para AS400, concentrado en productos premium de alto valor.
- La invisibilidad del catálogo no es marginal: los Ghost SKUs están activos en la operación física y no solo en el maestro de datos.
- Se detectó movilización material de mercancía bajo estos códigos, con alcance multirruta y presencia en una gran cantidad de tiendas.
- Existe superposición entre el flujo de Ghost SKUs y las tiendas previamente identificadas con mayor severidad de discrepancia.
- Esto confirma que la opacidad de catálogo y la pérdida operativa no están ocurriendo en circuitos separados, sino convergiendo en nodos comunes de riesgo.

### Interpretación metodológica
El análisis no demuestra por sí mismo intencionalidad individual.  
Sí demuestra, en cambio, una vulnerabilidad estructural de trazabilidad con impacto económico material y manifestación operativa activa.

### Decisión analítica
La debilidad de catálogo debe tratarse como una capa de riesgo prioritaria dentro de la investigación, no como un defecto administrativo secundario.

## Próximo paso recomendado

La siguiente etapa del proyecto debe enfocarse en:

1. **Cruzar SKUs invisibles con usuarios autorizadores y rutas operativas**
2. **Medir la intersección entre inventario invisible, severidad de discrepancia y ventana horaria atípica**
3. **Construir una matriz de riesgo unificada por tienda, ruta y flujo económico**
4. **Preparar una narrativa ejecutiva consolidada para escalamiento a comité**

La pregunta ya no es si existe opacidad estructural, sino **qué parte de esa opacidad coincide con pérdida material y bajo qué controles ocurre**.